# EDD Phase 3 Week 6 — RETFound + EfficientNet-V2-S Ensemble

**5-Class Retinal Disease Classification for Production Accuracy**

Classes: Normal | DR | Glaucoma | Cataract | AMD

---
## Setup
Before running this notebook:
1. Enable **Internet** (right sidebar → Settings → Internet ON)
2. Add **HF_TOKEN** via Add-ons → Secrets
3. Add datasets: APTOS 2019 (Featured, 2928 teams), Eye Diseases Classification
4. Set GPU Accelerator: **GPU T4 x2** or P100


## Cell 1 — Hugging Face Authentication

In [ ]:
# ── Hugging Face Authentication ─────────────────────────────────────────────
# This MUST run first before any model download
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = hf_token
print("✅ Hugging Face authenticated")


## Cell 2 — Install/Verify Dependencies

In [ ]:
import subprocess, sys
# Ensure required packages are present
packages = [
    'albumentations>=1.3.0',
    'timm>=0.9.0',
    'huggingface_hub>=0.20.0',
    'safetensors',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print('✅ Dependencies ready')


## Cell 3 — Verify RETFound Access

In [ ]:
from huggingface_hub import hf_hub_download
import os

hf_token = os.environ.get('HF_TOKEN')
try:
    path = hf_hub_download(
        repo_id='YukunZhou/RETFound_mae_natureCFP',
        filename='README.md',
        token=hf_token
    )
    print(f'✅ RETFound access confirmed')
    USE_PRIMARY = True
except Exception as e:
    print(f'⚠️  Primary repo not accessible: {e}')
    print('→ Will use public fallback: iszt/RETFound_mae_meh')
    USE_PRIMARY = False


## Cell 4 — Upload & Import Main Script

> **Upload `kaggle_phase3_week6.py`** to your notebook session using the
> **File → Upload** button in the Kaggle editor sidebar, or add it as a dataset.
> Then run this cell.

In [ ]:
import sys
from pathlib import Path

# If uploaded directly to session
sys.path.insert(0, '/kaggle/working')
sys.path.insert(0, '/kaggle/input')

# Verify the script is importable
script_path = Path('/kaggle/working/kaggle_phase3_week6.py')
if not script_path.exists():
    # Check input datasets
    for p in Path('/kaggle/input').rglob('kaggle_phase3_week6.py'):
        import shutil
        shutil.copy(p, script_path)
        print(f'Copied from {p}')
        break

assert script_path.exists(), 'Please upload kaggle_phase3_week6.py to /kaggle/working/'
print(f'✅ Script found at {script_path}')


## Cell 5 — Import All Modules

In [ ]:
from kaggle_phase3_week6 import (
    CFG, DEVICE, CLASS_NAMES, CLASS_TO_IDX,
    seed_everything,
    build_combined_dataframe,
    build_dataloaders,
    compute_class_weights,
    load_retfound_weights,
    build_retfound,
    build_efficientnet,
    train_phase_a,
    train_phase_b,
    train_efficientnet,
    ensemble_evaluate,
    plot_confusion_matrix,
    plot_training_curves,
    FocalLoss,
)
import os, torch
print(f'✅ All imports successful')
print(f'Device  : {DEVICE}')
print(f'Classes : {CLASS_NAMES}')


## Cell 6 — Build Combined Dataset

Loads ODIR-2019 + RFMiD + APTOS 2019 + Eye Diseases Classification.
Drops 'Other', Hypertension, and Myopia rows. Reports class distribution.

In [ ]:
df_train, df_val, df_test = build_combined_dataframe(
    CFG['odir_root'],
    CFG['rfmid_root'],
    CFG['aptos_root'],
    CFG['eyedis_root'],
)
class_weights = compute_class_weights(df_train)
print(f'\n✅ Dataset ready: {len(df_train)} train / {len(df_val)} val / {len(df_test)} test')


## Cell 7 — Build DataLoaders (224×224 for RETFound)

In [ ]:
loader_train, loader_val, loader_test = build_dataloaders(
    df_train, df_val, df_test,
    img_size=CFG['img_size'],
    batch_size=CFG['phase_b_batch'],
)
print(f'✅ Loaders ready')
print(f'   Train batches : {len(loader_train)}')
print(f'   Val   batches : {len(loader_val)}')
print(f'   Test  batches : {len(loader_test)}')


## Cell 8 — Download RETFound Weights (~1.3 GB)

This will take 3–5 minutes on first run. Subsequent runs use the cached file.

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
CFG['retfound_file'] = 'RETFound_mae_natureCFP.pth'
weight_path = load_retfound_weights(hf_token=hf_token)
print(f'✅ Weights ready at: {weight_path}')


## Cell 9 — Build RETFound Model

In [ ]:
retfound = build_retfound(num_classes=CFG['num_classes'], weight_path=weight_path)
print('✅ RETFound model ready')


## Cell 10 — Phase A: Linear Probe (5 epochs)

Backbone frozen. Only the classification head is trained.
Expected time: **~15–20 min** on T4.

In [ ]:
retfound = train_phase_a(retfound, loader_train, loader_val, class_weights)
print('✅ Phase A complete')


## Cell 11 — Phase B: Full Fine-Tune (up to 30 epochs)

All layers unfrozen with layer-wise LR decay.
Expected time: **~2–3 hours** on T4. Early stopping patience = 7.

> ⚠️ This is the long cell. Kaggle will keep running even if you close the browser tab.

In [ ]:
retfound, retfound_val_acc, ret_history = train_phase_b(
    retfound, loader_train, loader_val, class_weights
)
print(f'✅ Phase B complete — Best Val Bal.Acc: {retfound_val_acc:.4f}')


## Cell 12 — Load Best RETFound Checkpoint

In [ ]:
import torch
from pathlib import Path

best_ckpt = Path(CFG['ckpt_dir']) / 'retfound_best.pth'
state = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
retfound.load_state_dict(state['model_state'])
print(f"✅ Loaded RETFound Epoch {state['epoch']} (Val Bal.Acc = {state['bal_acc']:.4f})")


## Cell 13 — Train EfficientNet-V2-S (up to 40 epochs)

288×288 images, light Mixup (prob=0.3), Focal Loss.
Expected time: **~1 hour** on T4.

In [ ]:
CFG['effnet_model'] = 'efficientnetv2_rw_s'
effnet = build_efficientnet(num_classes=CFG['num_classes'])
effnet, effnet_val_acc = train_efficientnet(effnet, df_train, df_val, class_weights)
print(f'✅ EfficientNet training complete — Best Val Bal.Acc: {effnet_val_acc:.4f}')


## Cell 14 — Load Best EfficientNet Checkpoint

In [ ]:
ckpt_eff = torch.load(
    Path(CFG['ckpt_dir']) / 'effnet_best.pth',
    map_location=DEVICE, weights_only=False
)
effnet.load_state_dict(ckpt_eff['model_state'])
print(f"✅ Loaded EffNet Epoch {ckpt_eff['epoch']} (Val Bal.Acc = {ckpt_eff['bal_acc']:.4f})")


## Cell 15 — Ensemble Evaluation on Test Set

Combines RETFound (65%) + EfficientNet-V2-S (35%) softmax outputs.
This is your final production score.

In [ ]:
bal_acc, macro_f1, preds, labels, ensemble_probs = ensemble_evaluate(
    retfound, effnet, loader_test, df_test
)


## Cell 16 — Save Plots

In [ ]:
from pathlib import Path

plot_confusion_matrix(
    labels, preds,
    'Week 6 Ensemble — Test Confusion Matrix',
    Path(CFG['result_dir']) / 'confusion_matrix.png'
)
plot_training_curves(
    ret_history,
    'RETFound Fine-Tuning Curves',
    Path(CFG['result_dir']) / 'retfound_curves.png'
)

from IPython.display import Image as IPImage, display
display(IPImage(Path(CFG['result_dir']) / 'confusion_matrix.png'))
display(IPImage(Path(CFG['result_dir']) / 'retfound_curves.png'))


## Cell 17 — Final Summary

In [ ]:
import json
summary = {
    'phase':             'Phase 3 Week 6',
    'retfound_val_acc':  float(retfound_val_acc),
    'effnet_val_acc':    float(effnet_val_acc),
    'ensemble_bal_acc':  float(bal_acc),
    'ensemble_macro_f1': float(macro_f1),
    'classes':           CLASS_NAMES,
}
print(json.dumps(summary, indent=2))

summary_path = Path(CFG['result_dir']) / 'summary_metrics.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(f'\n✅ Summary saved to {summary_path}')
